In [ ]:
#this code is without streamlit or postgre sql, it runs a local OpenCV window and prints JSON telemetry to the terminal.
import cv2
import numpy as np
from ultralytics import YOLO
import json
import time
import datetime

# --- CONFIGURATION ---
MODEL_V3_PATH = r"model path\best.pt"  # Replace with your V3 specialist model path
MODEL_COCO_PATH = "yolov8s.pt"
VIDEO_PATH = r"video path"

CONF_V3 = 0.15
CONF_COCO = 0.15
# ---------------------


def boxes_overlap(box1, box2):
    return not (box1[2] < box2[0] or box1[0] > box2[2] or box1[3] < box2[1] or box1[1] > box2[3])


def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(c1, c2):
    return np.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)


class GlobalTracker:
    def __init__(self, max_age=45, max_dist=150):
        self.tracks = {}
        self.next_id = 1
        self.max_age = max_age
        self.max_dist = max_dist

    def update(self, current_boxes):
        matched_ids = set()
        for box in current_boxes:
            cx, cy = get_center(box)
            best_id, min_dist = None, float('inf')
            for tid, trk in self.tracks.items():
                if tid in matched_ids:
                    continue
                dist = get_distance((cx, cy), trk['center'])
                if boxes_overlap(box, trk['box']) or dist < self.max_dist:
                    if dist < min_dist:
                        min_dist, best_id = dist, tid

            if best_id is not None:
                self.tracks[best_id]['box'] = box
                self.tracks[best_id]['center'] = (cx, cy)
                self.tracks[best_id]['age'] = 0
                matched_ids.add(best_id)
            else:
                self.tracks[self.next_id] = {'box': box, 'center': (
                    cx, cy), 'age': 0, 'status': 'WAITING'}
                matched_ids.add(self.next_id)
                self.next_id += 1

        to_delete = []
        for tid in self.tracks:
            if tid not in matched_ids:
                self.tracks[tid]['age'] += 1
                if self.tracks[tid]['age'] > self.max_age:
                    to_delete.append(tid)
        for tid in to_delete:
            del self.tracks[tid]
        return self.tracks


# Load Models
model_v3 = YOLO(MODEL_V3_PATH)
model_coco = YOLO(MODEL_COCO_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps):
    fps = 30.0

truck_tracker = GlobalTracker(max_age=60, max_dist=150)
excavator_tracker = GlobalTracker(max_age=9000, max_dist=400)

# --- PAYLOAD & STATE VARIABLES ---
total_loads = 0
is_dumping = False
dumping_frames = 0
REQUIRED_DUMP_FRAMES = 1

last_known_bucket = None
bucket_missing_frames = 0

load_cooldown_timer = 0
COOLDOWN_MAX = 90

current_activity = "IDLE"
idle_timer = 0
IDLE_THRESHOLD = 300

frame_count = 0
total_active_frames = 0
total_idle_frames = 0
# -------------------------

print("--- Running Ultimate OpenCV Tracker (With JSON Telemetry) ---")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    annotated_frame = frame.copy()
    frame_area = frame.shape[0] * frame.shape[1]
    frame_height = frame.shape[0]

    raw_excavator_boxes = []
    raw_truck_boxes = []
    bucket_center = None

    if load_cooldown_timer > 0:
        load_cooldown_timer -= 1

    # ==========================================
    # 1. RUN BRAIN 1 (V3 Specialist)
    # ==========================================
    res_v3 = model_v3.predict(frame, conf=CONF_V3, verbose=False)

    if res_v3[0].boxes is not None:
        v3_boxes = res_v3[0].boxes.xyxy.cpu().numpy()
        v3_clss = res_v3[0].boxes.cls.cpu().numpy()

        for box, cls in zip(v3_boxes, v3_clss):
            x1, y1, x2, y2 = [int(x) for x in box]

            if int(cls) == 1:
                ex_area = (x2 - x1) * (y2 - y1)
                if ex_area > (frame_area * 0.03):
                    raw_excavator_boxes.append([x1, y1, x2, y2])
            elif int(cls) == 2:
                raw_truck_boxes.append([x1, y1, x2, y2])

        # --- BUCKET EXTRACTION ---
        bucket_found_this_frame = False
        if res_v3[0].masks is not None:
            for i, cls in enumerate(v3_clss):
                if int(cls) == 0:
                    mask = res_v3[0].masks.data[i].cpu().numpy()
                    mask = cv2.resize(mask, (frame.shape[1], frame.shape[0]))
                    M = cv2.moments(mask)
                    if M["m00"] != 0:
                        bx, by = int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])
                        bucket_center = (bx, by)
                        last_known_bucket = bucket_center
                        bucket_missing_frames = 0
                        bucket_found_this_frame = True

                        cv2.circle(annotated_frame, (bx, by),
                                   8, (0, 0, 255), -1)
                        cv2.putText(annotated_frame, "BUCKET",
                                    (bx+10, by), 0, 0.6, (0, 0, 255), 2)
                    break

        # SHRT GHOST MEMORY: Snaps out after 15 frames
        if not bucket_found_this_frame and last_known_bucket is not None:
            bucket_missing_frames += 1
            if bucket_missing_frames < 15:
                bucket_center = last_known_bucket
                cv2.circle(annotated_frame, bucket_center,
                           8, (0, 165, 255), -1)
            else:
                last_known_bucket = None

    # ==========================================
    # 2. GLUE EXCAVATORS & TRACK
    # ==========================================
    glued_excavator_boxes = []
    for raw_box in raw_excavator_boxes:
        merged = False
        for i, g_box in enumerate(glued_excavator_boxes):
            dist = get_distance(get_center(raw_box), get_center(g_box))
            if boxes_overlap(raw_box, g_box) or dist < 400:
                new_x1 = min(raw_box[0], g_box[0])
                new_y1 = min(raw_box[1], g_box[1])
                new_x2 = max(raw_box[2], g_box[2])
                new_y2 = max(raw_box[3], g_box[3])
                glued_excavator_boxes[i] = [new_x1, new_y1, new_x2, new_y2]
                merged = True
                break
        if not merged:
            glued_excavator_boxes.append(raw_box)

    active_excavators = excavator_tracker.update(glued_excavator_boxes)
    current_excavator_zones = []
    primary_excavator_id = 1

    for ex_id, ex_data in active_excavators.items():
        primary_excavator_id = ex_id
        ex1, ey1, ex2, ey2 = ex_data['box']
        current_excavator_zones.append(ex_data['box'])
        cv2.rectangle(annotated_frame, (ex1, ey1), (ex2, ey2), (255, 0, 0), 2)
        cv2.rectangle(annotated_frame, (ex1, ey1-35),
                      (ex1+200, ey1), (255, 0, 0), -1)
        cv2.putText(
            annotated_frame, f"EXCAVATOR {ex_id}", (ex1+5, ey1-5), 0, 0.7, (255, 255, 255), 2)

    # ==========================================
    # 3. RUN BRAIN 2 (Waiting Trucks)
    # ==========================================
    res_coco = model_coco.predict(
        frame, classes=[7], conf=CONF_COCO, verbose=False)

    if res_coco[0].boxes is not None:
        coco_boxes = res_coco[0].boxes.xyxy.cpu().numpy()
        for box in coco_boxes:
            cx1, cy1, cx2, cy2 = [int(x) for x in box]
            coco_rect = [cx1, cy1, cx2, cy2]
            box_area = (cx2 - cx1) * (cy2 - cy1)

            if box_area < (frame_area * 0.001) or box_area > (frame_area * 0.05):
                continue

            is_hallucination = False
            for ex_zone in current_excavator_zones:
                if boxes_overlap(coco_rect, ex_zone):
                    is_hallucination = True
                    break
            for v3_box in raw_truck_boxes:
                if boxes_overlap(coco_rect, v3_box):
                    is_hallucination = True
                    break

            if not is_hallucination:
                raw_truck_boxes.append(coco_rect)

    # ==========================================
    # 4. SMART GLUER FOR TRUCKS
    # ==========================================
    glued_truck_boxes = []
    for raw_box in raw_truck_boxes:
        merged = False
        for i, g_box in enumerate(glued_truck_boxes):
            dist = get_distance(get_center(raw_box), get_center(g_box))
            if boxes_overlap(raw_box, g_box) and dist < 200:
                new_x1 = min(raw_box[0], g_box[0])
                new_y1 = min(raw_box[1], g_box[1])
                new_x2 = max(raw_box[2], g_box[2])
                new_y2 = max(raw_box[3], g_box[3])
                glued_truck_boxes[i] = [new_x1, new_y1, new_x2, new_y2]
                merged = True
                break
        if not merged:
            glued_truck_boxes.append(raw_box)

    active_global_tracks = truck_tracker.update(glued_truck_boxes)
    active_truck_box = None

    for track_id, track_data in active_global_tracks.items():
        tx1, ty1, tx2, ty2 = track_data['box']
        trk_rect = [tx1, ty1, tx2, ty2]
        is_active_zone = False

        trk_area = (tx2 - tx1) * (ty2 - ty1)
        if trk_area > (frame_area * 0.05):
            is_active_zone = True

        if not is_active_zone:
            for ex_zone in current_excavator_zones:
                ex1, ey1, ex2, ey2 = ex_zone
                loading_zone = [ex1 - 250, ey1 - 150, ex2 + 250, ey2 + 150]
                if boxes_overlap(trk_rect, loading_zone):
                    is_active_zone = True
                    break

        track_data['status'] = 'ACTIVE' if is_active_zone else 'WAITING'

        # Grab the active truck box for payload math
        if is_active_zone and track_data['age'] == 0:
            active_truck_box = track_data['box']

        if is_active_zone:
            color = (0, 255, 0)
            status_text = f"ACTIVE TRUCK {track_id}"
            thickness = 3
        else:
            color = (0, 255, 255)
            status_text = f"WAITING TRUCK {track_id}"
            thickness = 2

        if track_data['age'] > 0:
            color = (0, 100, 0) if is_active_zone else (0, 100, 100)

        cv2.rectangle(annotated_frame, (tx1, ty1),
                      (tx2, ty2), color, thickness)
        cv2.rectangle(annotated_frame, (tx1, ty1-35),
                      (tx1+200, ty1), color, -1)
        cv2.putText(annotated_frame, status_text,
                    (tx1+5, ty1-5), 0, 0.6, (0, 0, 0), 2)

    # ==========================================
    # 5. PAYLOAD MATH (Fast Trigger & Clean Exit)
    # ==========================================
    activity_this_frame = None

    if bucket_center:
        bx, by = bucket_center

        if active_truck_box:
            tx1, ty1, tx2, ty2 = active_truck_box

            # The tight box you requested
            dump_top = ty1 - 100
            dump_left = tx1 - 100
            dump_right = tx2 + 100

            if dump_left < bx < dump_right and dump_top < by < ty2:
                dumping_frames += 1
                if dumping_frames >= REQUIRED_DUMP_FRAMES:
                    is_dumping = True
                    activity_this_frame = "DUMPING PAYLOAD"
                else:
                    activity_this_frame = "SWINGING"
            else:
                dig_threshold = ty1 + 50
                if by > dig_threshold:
                    activity_this_frame = "DIGGING"
                else:
                    activity_this_frame = "SWINGING"

                # Triggers the count the exact moment the bucket lifts above 'dump_top'
                if is_dumping:
                    if load_cooldown_timer == 0:
                        total_loads += 1
                        load_cooldown_timer = COOLDOWN_MAX
                    is_dumping = False
                dumping_frames = 0
        else:
            if by > (frame_height * 0.6):
                activity_this_frame = "DIGGING"
            else:
                activity_this_frame = "SWINGING"
    else:
        # Dust Cloud Catcher
        if is_dumping:
            if load_cooldown_timer == 0:
                total_loads += 1
                load_cooldown_timer = COOLDOWN_MAX
            is_dumping = False
        dumping_frames = 0

    if activity_this_frame is not None:
        current_activity = activity_this_frame
        idle_timer = 0
    else:
        idle_timer += 1
        if idle_timer > IDLE_THRESHOLD:
            current_activity = "IDLE"

    # ==========================================
    # 6. JSON 
    # ==========================================
    if current_activity == "IDLE":
        total_idle_frames += 1
        status_state = "IDLE"
    else:
        total_active_frames += 1
        status_state = "ACTIVE"

    # Print JSON to Terminal once every real-world second
    if frame_count % int(fps) == 0:
        total_seconds = frame_count / fps
        active_seconds = total_active_frames / fps
        idle_seconds = total_idle_frames / fps

        if frame_count > 0:
            utilization_percent = (total_active_frames / frame_count) * 100
        else:
            utilization_percent = 0.0

        payload = {
            "frame_id": frame_count,
            "equipment_id": f"EX-{primary_excavator_id:03d}",
            "equipment_class": "excavator",
            "timestamp": str(datetime.timedelta(seconds=int(total_seconds))) + f".{int((total_seconds % 1) * 1000):03d}",
            "utilization": {
                "current_state": status_state,
                "current_activity": current_activity,
                "motion_source": "arm_only"
            },
            "time_analytics": {
                "total_tracked_seconds": round(total_seconds, 1),
                "total_active_seconds": round(active_seconds, 1),
                "total_idle_seconds": round(idle_seconds, 1),
                "utilization_percent": round(utilization_percent, 1)
            }
        }

        # Clear the terminal slightly and print the JSON
        print("\n--- NEW TELEMETRY EVENT ---")
        print(json.dumps(payload, indent=2))

    # ==========================================
    # 7. DASHBOARD UI (Standard OpenCV Window)
    # ==========================================
    cv2.rectangle(annotated_frame, (10, 10), (360, 75), (0, 0, 0), -1)

    if current_activity == "DUMPING PAYLOAD":
        act_color = (0, 255, 255)
    elif current_activity == "SWINGING":
        act_color = (255, 165, 0)
    elif current_activity == "DIGGING":
        act_color = (0, 255, 0)
    else:
        act_color = (255, 255, 255)

    cv2.putText(annotated_frame,
                f"ACT: {current_activity}", (20, 35), 0, 0.6, act_color, 2)
    cv2.putText(annotated_frame,
                f"LOADS: {total_loads}", (20, 65), 0, 0.8, (255, 255, 255), 2)

    if load_cooldown_timer > 0:
        cv2.putText(annotated_frame, "COUNTER LOCKED",
                    (20, 100), 0, 0.6, (0, 0, 255), 2)

    cv2.imshow("Production Mining Tracker", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

--- Running Ultimate OpenCV Tracker (With JSON Telemetry) ---

--- NEW TELEMETRY EVENT ---
{
  "frame_id": 29,
  "equipment_id": "EX-001",
  "equipment_class": "excavator",
  "timestamp": "0:00:00.967",
  "utilization": {
    "current_state": "IDLE",
    "current_activity": "IDLE",
    "motion_source": "arm_only"
  },
  "time_analytics": {
    "total_tracked_seconds": 1.0,
    "total_active_seconds": 0.0,
    "total_idle_seconds": 1.0,
    "utilization_percent": 0.0
  }
}

--- NEW TELEMETRY EVENT ---
{
  "frame_id": 58,
  "equipment_id": "EX-001",
  "equipment_class": "excavator",
  "timestamp": "0:00:01.935",
  "utilization": {
    "current_state": "ACTIVE",
    "current_activity": "DIGGING",
    "motion_source": "arm_only"
  },
  "time_analytics": {
    "total_tracked_seconds": 1.9,
    "total_active_seconds": 0.4,
    "total_idle_seconds": 1.5,
    "utilization_percent": 20.7
  }
}

--- NEW TELEMETRY EVENT ---
{
  "frame_id": 87,
  "equipment_id": "EX-001",
  "equipment_class": "exc